In [ ]:
# Install latest version from local files (only necessary after pulling updated version from Git).
# Restart Jupyter kernel when install has finished.
%pip install -e ..

In [1]:
# Some Windows installations require explicitly installing mne (required only once; restart Jupyter kernel after installing).
%pip install mne[full] mne_bids

Note: you may need to restart the kernel to use updated packages.


In [2]:
from biopap.pipeline import PipelineStep, Pipeline

from biopap.modules.load_data import LoadData
from biopap.modules.anonymize import Anonymize
from biopap.modules.rereference import Rereference
from biopap.modules.eog_channels import EOGChannels
from biopap.modules.iir_filter import IIRFilter
from biopap.modules.downsample import Downsample
from biopap.modules.segmentation import Segmentation
from biopap.modules.plot import Plot

In [3]:
from pathlib import Path
from pprint import pprint

In [4]:
# To avoid issues with local settings and GitHub updates, create a small notebook
# called 'my_config.ipynb' to hold variables that are likely to change locally. 
# In the same folder as this file, there is an example notebook called
# 'example.my_config.ipynb' that can serve as a template.

config_notebook = './my_config.ipynb'

if Path(config_notebook).exists():
    %run {config_notebook}
else:
    data_file = '/biopap/data/example.bdf'
    config_file = './data/pipeline.yaml'

if not Path(config_file).exists:
    print(f"Config file {config_file} does not exist.")

if not Path(data_file).exists:
    print(f"Data file {data_file} does not exist.")

### About modules

When instantiated, all processing modules receive the data to be processed as RawEDF-type object, which is assigned to an attribute labeled `raw`. Once done processing, the processed data is available as an attribute labeled `processed` of the module. The processed data has the same format as the raw input, so the output of one module can become the input of the next, thus allowing you to easily chain modules together.

The only exception is the Load module, which reads the initial data from disk without any further processing, and therefore only has raw data.

### Loading data

In [5]:
# Read the data file and load it into memory.
load = LoadData(file_path=data_file, logger=None)

Extracting EDF parameters from /home/matty/ResearchEngineers/bioPAP/data/CBN1BD071212-Cropped.bdf...
BDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 3276799  =      0.000 ...  1600.000 secs...


In [6]:
ref = Rereference(raw=load.raw, ref_channels=['EXG1', 'EXG2'], logger=None)
ref.processed.save('./examples/test_output.fif', overwrite=True)

EEG channel type selected for re-referencing
Applying a custom ('EEG',) reference.
Overwriting existing file.
Writing /home/matty/ResearchEngineers/bioPAP/BioPAP/examples/examples/test_output.fif
Overwriting existing file.


/tmp/ipykernel_55467/70508397.py:2: RuntimeWarning: This filename (/home/matty/ResearchEngineers/bioPAP/BioPAP/examples/examples/test_output.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  ref.processed.save('./examples/test_output.fif', overwrite=True)


Closing /home/matty/ResearchEngineers/bioPAP/BioPAP/examples/examples/test_output.fif
[done]


[PosixPath('/home/matty/ResearchEngineers/bioPAP/BioPAP/examples/examples/test_output.fif')]

In [7]:
# Display basic info
load.raw.info

<Info | 9 non-empty values
 bads: []
 ch_names: Fp1, AF7, AF3, F1, F3, F5, F7, FT7, FC5, FC3, FC1, C1, C3, C5, ...
 chs: 79 EEG, 1 Stimulus
 custom_ref_applied: True
 highpass: 0.0 Hz
 lowpass: 417.0 Hz
 meas_date: 2012-12-07 12:14:08 UTC
 nchan: 80
 projs: []
 sfreq: 2048.0 Hz
 subject_info: <subject_info | his_id: CBN1BD071212>
>

In [8]:
# Show channel info
load.raw.describe()

<RawEDF | CBN1BD071212-Cropped.bdf, 80 x 3276800 (1600.0 s), ~1.95 GiB, data loaded>
ch  name    type  unit        min         Q1     median         Q3        max
 0  Fp1     EEG   µV     -8210.47   -7014.50   -5796.32   -4944.90   -4088.73
 1  AF7     EEG   µV     -5182.99   -4490.99   -3224.53   -2621.01    -705.01
 2  AF3     EEG   µV     -3839.37   -2825.34   -2320.54   -2182.15   -1578.73
 3  F1      EEG   µV      5149.87    7527.89    9038.12    9376.84    9680.09
 4  F3      EEG   µV      4753.87    6269.21    7508.58    7953.95    8306.02
 5  F5      EEG   µV     -2695.96   -1189.11      59.17     406.22     801.45
 6  F7      EEG   µV      6774.92    7590.42    7809.75    8222.08    8792.61
 7  FT7     EEG   µV      -436.25    1150.19    1579.23    1744.43    1983.65
 8  FC5     EEG   µV      -642.50    2091.01    3546.92    4112.18    4685.41
 9  FC3     EEG   µV     -5455.79   -2442.68     395.64    1958.40    3028.04
10  FC1     EEG   µV      2810.28    5521.13    7058.35  

In [9]:
# use the Plot-module to inspect the data
plt = Plot(raw=load.raw, title="Initial state")

TypeError: Plot.__init__() missing 1 required positional argument: 'logger'

### Anonymization

In [ ]:
# defaults: Participant='0'; Experimenter='mne_anonymize'
ano = Anonymize(raw=load.raw, daysback=100, experimenter='J. Jansen', participant_id='AB00123')
ano.processed.info

### Rereferencing

In [ ]:
ref = Rereference(raw=load.raw, ref_channels=['EXG1', 'EXG2'], logger=None, save='./examples/test_output.fif')
ref.processed.info

### hEOG & vEOG

In [ ]:
eog = EOGChannels(
    raw=ref.processed,
    ref_channels=['hEOG', 'vEOG'],
    anode_channels=['EXG3','EXG5'],
    cathode_channels=['EXG4','EXG6'])

In [ ]:
print(eog.processed.info)

### Filtering

In [ ]:
flt = IIRFilter(
    raw=eog.processed,
    l_freq=0.1,
    h_freq=35)

In [ ]:
plt = Plot(raw=flt.processed)

In [ ]:
# Alternatively, save a plot of the current state by specifying save folder. The screenshot will take the basename of the original
# source file, optionally taking a postfix to distinguish between multiple screenshots on separate points in the pipeline.
# This step is not interactive, so can also be used in a pipeline intended to run unattended.
# N.B., showing the interactive plot and saving cannot be done at the same time, but you can add multiple subsequent Plot-steps.
_ = Plot(raw=flt.processed, save_folder='./', save_postfix='IRR')

### Downsampling

In [ ]:
dwn = Downsample(
    raw=flt.processed,
    sfreq=500)

In [ ]:
print(dwn.processed)
print(dwn.events)